# KV Caching with Valkey

Caching the attention KV an LLM computes for a shared prompt prefix cuts cost
per request and shortens time-to-first-token. This walkthrough stores that KV
in Valkey so it can be reused across vLLM instances.

When an LLM answers a prompt, it first computes attention key/value tensors for
every input token. This is the prefill step: its cost is proportional to the
prompt length, and the result is normally thrown away after each request.
Whenever a later request shares a prefix with an earlier one, prefill recomputes
the same KV from scratch.

Caching that KV and reloading it skips the recompute: you pay for prefill once
and reuse it. Shared prefixes are common in practice:

- **Multi-turn chat:** each turn typically resends much of the conversation so
  far, so a later turn shares a long prefix with the earlier ones. How much is
  resent depends on the application, but something almost always is.
- **A common preamble:** a system prompt, tool and function definitions, MCP
  context, skills, or other instructions sit in front of many requests.
- **RAG and agentic memory:** retrieved passages or recalled memories are often
  reused across requests, depending on the retrieval strategy.

[LMCache](https://github.com/LMCache/LMCache) is the KV-cache layer that plugs
into vLLM and does this. It can store KV in several places, including local
memory, disk, or a remote backend; this walkthrough uses Valkey as that
backend. vLLM's own built-in prefix cache is limited to what one engine can hold
and is not shared with other engines. Valkey is a good fit when several vLLM
instances must read one cache, and that cache needs to be larger than any single
host can hold. The instances sharing a cache must run the same model and tensor
configuration, since the cached KV is specific to both.

## Prerequisites

- **A container runtime with Compose** (Docker Compose or compatible). The
  stack is three containers: Valkey plus two vLLM replicas.
- **Memory for two model servers.** Give your container runtime **at least
  10 GiB** of RAM. Each vLLM replica needs roughly 3–4 GiB. On a smaller
  machine, run one replica and skip the two-replica step near the end.
- **Python 3.10-3.13**

**The stack:**

| Component | Role |
| --- | --- |
| [vLLM](https://docs.vllm.ai) (CPU image) | Serves the model and owns the KV cache |
| [LMCache](https://docs.lmcache.ai) | Offloads/loads KV to the remote store |
| [Valkey](https://valkey.io) | The shared KV-cache store |
| Model (`facebook/opt-125m`) | Small and ungated, so no GPU or token needed |

Install this notebook's client dependencies (from this directory):

```bash
python3 -m venv .venv
.venv/bin/python -m pip install -r requirements.txt
.venv/bin/python -m ipykernel install --user --name kv-caching-valkey
```

Then select the `kv-caching-valkey` kernel when you open this notebook.

> **CPU vs GPU:** this walkthrough runs entirely on CPU so anyone can follow it.
> A GPU speeds up the model's own compute. It does not change the caching
> behavior.

## Setup

Start the whole stack, Valkey and both vLLM replicas, with one command from
this directory:

```bash
docker compose up -d --wait
```

`--wait` blocks until every container reports healthy. **The first run is slow:**
the vLLM image is large and each replica compiles the model on startup, so give
it several minutes. Subsequent runs reuse the cached image and model weights.

### Pointing LMCache at Valkey

Wiring LMCache to Valkey is one line, the `remote_url` in
[`lmcache_config.yaml`](lmcache_config.yaml), which both replicas load:

```yaml
remote_url: "valkey://valkey:6379"
```

That is the whole integration: name the Valkey host, and LMCache stores and
loads KV there. `docker-compose.yml` mounts this file into each vLLM container
and sets `LMCACHE_CONFIG_FILE` so vLLM's LMCache connector picks it up. See the
[LMCache Valkey/Redis backend docs](https://docs.lmcache.ai/kv_cache/redis.html)
for the other backends and options.

> **Security:** This stack uses no authentication or TLS for simplicity.
> For any non-localhost deployment, enable authentication and TLS.
> See the [Valkey security documentation](https://valkey.io/topics/security/).

In [ ]:
# A tiny bit of shared setup: talk to each replica over its OpenAI-compatible
# HTTP API, and read cache signals straight from Valkey via the CLI.
import subprocess
from openai import OpenAI

MODEL = "facebook/opt-125m"

# The two vLLM replicas, each mapped to a host port by docker-compose.yml.
replica_a = OpenAI(base_url="http://localhost:8001/v1", api_key="not-needed")
replica_b = OpenAI(base_url="http://localhost:8002/v1", api_key="not-needed")


def valkey_cli(*args):
    """Run a valkey-cli command inside the Valkey container and return stdout."""
    out = subprocess.run(
        ["docker", "exec", "valkey", "valkey-cli", *args],
        capture_output=True, text=True,
    )
    return out.stdout.strip()


def dbsize():
    """Number of keys in Valkey. Each cached KV chunk is stored as keys here."""
    return int(valkey_cli("DBSIZE"))


def keyspace_hits():
    """Cumulative Valkey keyspace hits, increments when a stored key is read."""
    info = valkey_cli("INFO", "stats")
    for line in info.splitlines():
        if line.startswith("keyspace_hits:"):
            return int(line.split(":", 1)[1])
    return 0


print("Setup ready.")

Confirm the whole stack is live before going further: Valkey answers `PING`, and
both replicas list the model on `/v1/models`.

In [ ]:
print("valkey ping :", valkey_cli("PING"))
print("replica A   :", [m.id for m in replica_a.models.list().data])
print("replica B   :", [m.id for m in replica_b.models.list().data])

## Example: a multi-turn chat

We walk through one scenario start to finish, so you can map it onto your own
use. We hold a short multi-turn conversation with replica A and watch the KV
cache in Valkey between turns. Then we send the same conversation to replica B
to prove the cache is actually shared.

We keep the conversation in an explicit `messages` list and grow it turn by
turn. That is what makes each turn share the previous turn's prefix, the same
thing that happens in any real chat UI.

In [ ]:
SYSTEM_PROMPT = (
    "You are a concise assistant that answers questions about world geography. "
    "Keep every answer to a single short sentence."
)

# The conversation so far. We append to this between turns.
messages = [{"role": "system", "content": SYSTEM_PROMPT}]


def ask(client, question, conversation):
    """Send the whole conversation + a new question, return the answer text."""
    conversation.append({"role": "user", "content": question})
    resp = client.chat.completions.create(
        model=MODEL, messages=conversation, max_tokens=32, temperature=0.0,
    )
    answer = resp.choices[0].message.content
    conversation.append({"role": "assistant", "content": answer})
    return answer

### Turn 1: cold

Nothing is cached yet. Replica A has to prefill the full prompt (system prompt +
first question), computing KV for every token from scratch. What gets cached is
exactly that KV: the attention key/value tensors for the prompt's tokens, split
into fixed-size chunks. As replica A prefills, LMCache writes those KV chunks to
Valkey.

We record Valkey's key count before and after. A cold turn should make it
**grow**: the freshly computed KV blocks have landed in Valkey.

In [ ]:
before = dbsize()
answer1 = ask(replica_a, "What is the capital of France?", messages)
after = dbsize()

print("Q1 answer     :", answer1)
print("Valkey dbsize :", before, "->", after)
print("KV blocks stored in Valkey:", after - before)

### Turn 2: warm, shared prefix

Now the follow-up. The conversation we send is `system + Q1 + A1 + Q2`, so its
first three parts are byte-for-byte the prefix replica A just prefilled. Only
the new question at the end is new.

**How the hit works.** LMCache splits the token sequence into fixed-size chunks
(`chunk_size: 256`) and keys each chunk by a hash of the tokens in that chunk
and every chunk before it. The Valkey key is a pure function of
`(model, world_size, worker_id, chunk_hash, dtype)`
(`vllm@<model>@<world>@<worker>@<hash>` on the wire), so two requests that start
with the same tokens produce the identical key. A write by one is a hit for the
other. The match is on an exact token prefix: identical tokens from the
start, up to the point where the two requests diverge. Turn 2 shares every token
of the system prompt and first exchange, so those chunks are a hit. LMCache
loads their KV from Valkey and vLLM only prefills the handful of new tokens in
Q2.

The signal is Valkey's `keyspace_hits`: reading the cached prefix chunks back
out increments it.

In [ ]:
hits_before = keyspace_hits()
answer2 = ask(replica_a, "What about Italy?", messages)
hits_after = keyspace_hits()

print("Q2 answer          :", answer2)
print("keyspace_hits      :", hits_before, "->", hits_after)
print("prefix chunks hit  :", hits_after - hits_before)
print()
print("A cache hit (delta > 0) means LMCache loaded the shared prefix's KV from")
print("Valkey instead of vLLM recomputing it.")

### A second replica hits the first replica's cache

Replica B has served no traffic and computed no KV of its own. We send it the
same conversation prefix replica A built up. The KV for that prefix lives in
Valkey, not inside replica A's process, so replica B finds it there and loads
it instead of prefilling from scratch.

Watch `keyspace_hits` again: the increment comes from **replica B** reading KV
that **replica A** computed. An in-process cache on replica A would be invisible
to replica B. Putting the cache in Valkey is what makes this cross-replica reuse
possible.

In [ ]:
# Replay replica A's exact conversation prefix (everything up to, but not
# including, replica A's last answer) as a fresh request to replica B.
shared_prefix = messages[:-1]

hits_before = keyspace_hits()
resp = replica_b.chat.completions.create(
    model=MODEL, messages=shared_prefix, max_tokens=32, temperature=0.0,
)
hits_after = keyspace_hits()

print("Replica B answer :", resp.choices[0].message.content)
print("keyspace_hits    :", hits_before, "->", hits_after)
print("prefix chunks B loaded from Valkey:", hits_after - hits_before)
print()
print("Replica B reused KV that replica A computed, through Valkey.")

## Cleanup

Tear down all three containers, Valkey and both vLLM replicas:

```bash
docker compose down
```

**What persists:** the KV cache lives in the Valkey container's memory, so it
survives an individual vLLM replica restarting. The cache is lost when
`docker compose down` removes the container, unless you mount a data volume and
enable append-only persistence to keep it across container recreation. The
`hf-cache` volume that holds the downloaded model weights does persist across
`down`/`up`, so later runs skip the re-download.

## What you learned

- **Why Valkey:** it holds the KV cache *outside* any single vLLM process, so
  one cache can be shared across many vLLM instances and can outlive any single
  one of them. Replica B reused a prefix that replica A had computed, something
  an in-process cache cannot do. (The cache is in-memory; it only survives a
  Valkey restart if you enable persistence.)
- **The multi-turn payoff:** each turn resends the prior turn's prefix, so the
  warm turn loaded that prefix's KV from Valkey instead of recomputing it.
- **The mechanism:** LMCache keys each KV chunk by a hash of its token prefix,
  so requests sharing an exact token prefix land on the same Valkey keys. A
  cold turn stores them, and later turns and other replicas read them back.

## Next steps

- **Scale out.** The natural continuation of the two-replica step: run *more*
  vLLM replicas, all sharing this one Valkey. Every replica warms the same cache
  and reads every other replica's prefixes, so the hit rate climbs as you add
  replicas. (This is scaling the vLLM fleet against one Valkey, not adding Valkey
  replicas.)
- **Take it to production.** See the
  [LMCache production deployment guide](https://docs.lmcache.ai/production/docker_deployment.html)
  for running LMCache under a served vLLM, and the
  [LMCache Valkey/Redis backend docs](https://docs.lmcache.ai/kv_cache/redis.html)
  for cluster mode, high availability, and tuning the remote store.